# 🔒 AMTTP Fraud Detection - BigQuery + Memgraph

**Works in both Google Colab Web UI and VS Code!**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides:
1. **BigQuery Integration** - Fetch real Ethereum transactions (FREE public dataset)
2. **GPU-Accelerated ML** - Fast scoring with CUDA (free T4 GPU in Colab)
3. **Memgraph Connection** - Graph-based fraud detection
4. **Real-time Scoring API** - Connect via ngrok

## 🚀 Quick Start

| Environment | Authentication | Google Drive |
|-------------|---------------|--------------|
| **Colab Web** | Auto popup | `drive.mount()` works |
| **VS Code** | `gcloud auth` or service account | Local storage |

### Colab Web UI (Recommended)
1. Open in Colab → Runtime → Change runtime type → T4 GPU
2. Run all cells - auth is automatic

### VS Code
1. Install gcloud CLI and run: `gcloud auth application-default login`
2. Or set `GOOGLE_APPLICATION_CREDENTIALS` to a service account JSON
3. Files save to local `./amttp_output/` folder

## 1️⃣ Install Dependencies

In [8]:
# Install required packages
!pip install -q google-cloud-bigquery db-dtypes pandas numpy pyarrow
!pip install -q xgboost lightgbm catboost
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q neo4j gqlalchemy pymgclient
!pip install -q fastapi uvicorn pyngrok
!pip install -q scikit-learn joblib

print("✅ Dependencies installed")

✅ Dependencies installed


In [9]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.2 GB


## 2️⃣ Connect Google Drive & Authenticate

In [10]:
# Mount Google Drive (works in Colab Web UI)
# For VS Code: Use Google Drive API instead

import os

# Detect environment
IN_COLAB_WEB = False
try:
    from google.colab import drive
    # Try to mount - works in Colab Web UI
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AMTTP_Fraud_Detection"
    os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
    print("✅ Google Drive mounted at /content/drive/MyDrive/")
    print(f"📂 Project folder: {DRIVE_PROJECT_DIR}")
    IN_COLAB_WEB = True
except Exception as e:
    # VS Code or local environment - use local storage
    print("📁 Running in VS Code (drive.mount not supported)")
    print("   Using local storage instead.")
    print()
    DRIVE_PROJECT_DIR = "./amttp_output"
    os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
    print(f"📂 Local project folder: {os.path.abspath(DRIVE_PROJECT_DIR)}")
    print()
    print("💡 To access Google Drive, use the Drive API:")
    print("   https://developers.google.com/drive/api/quickstart/python")

� Mounting Google Drive...


ValueError: mount failed

In [ ]:
# Authenticate with Google Cloud
# Colab Web UI: Shows popup
# VS Code: Use Python Cloud Client Library (gcloud CLI or service account)

from IPython.display import display, HTML
import os

def authenticate_gcp():
    """Authenticate with GCP - handles both Colab Web and VS Code."""
    
    # Method 1: Try Colab auth (works in Colab Web UI)
    try:
        from google.colab import auth
        auth.authenticate_user()
        print("✅ Authenticated via Colab (Web UI)")
        return True
    except Exception as e:
        pass
    
    # Method 2: Check for existing credentials (gcloud CLI or service account)
    from google.auth import default
    try:
        credentials, project = default()
        print(f"✅ Using existing credentials (project: {project})")
        return True
    except Exception:
        pass
    
    # Method 3: Show instructions for VS Code users
    print("=" * 60)
    print("🔐 Authentication Required (VS Code Environment)")
    print("=" * 60)
    print()
    print("Choose one of these options:")
    print()
    print("Option 1: Install & use gcloud CLI")
    print("  1. Download: https://cloud.google.com/sdk/docs/install")
    print("  2. Run in terminal: gcloud auth application-default login")
    print("  3. Re-run this cell")
    print()
    print("Option 2: Use Service Account")
    print("  1. Create a service account in GCP Console")
    print("  2. Download the JSON key file")
    print("  3. Set environment variable:")
    
    display(HTML('''
    <div style="padding: 15px; background: #fff3e0; border-radius: 8px; margin: 10px 0; border-left: 4px solid #ff9800;">
        <code style="background: #f5f5f5; padding: 5px 10px; border-radius: 4px;">
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/key.json"
        </code>
    </div>
    '''))
    
    print()
    print("Option 3: Run in Colab Web UI")
    display(HTML('''
    <div style="padding: 15px; background: #e3f2fd; border-radius: 8px; margin: 10px 0;">
        <p style="margin: 0;">
            👉 <a href="https://colab.research.google.com/" target="_blank" 
               style="color: #1976d2; font-weight: bold;">
               Open this notebook in Google Colab Web UI
            </a> for automatic authentication
        </p>
    </div>
    '''))
    
    return False

authenticated = authenticate_gcp()

In [ ]:
# Set your GCP project ID for BigQuery billing
# The public ethereum dataset is FREE to query, but you need a project for billing quota

PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}

# Find your project IDs:
!gcloud projects list --limit=5

⚠️ No API key - using public rate limit (slower)
   Get a FREE key: https://etherscan.io/apis


## 3️⃣ Fetch Ethereum Data from BigQuery

Using Google's public `crypto_ethereum` dataset - **FREE** to query!

In [ ]:
from google.cloud import bigquery
import pandas as pd
from datetime import datetime, timedelta

# Create BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Configuration
DAYS = 7  # @param {type:"integer"}
MIN_VALUE_ETH = 0.1  # @param {type:"number"}
SAMPLE_RATE = 0.01  # @param {type:"number"} 1% sample for cost efficiency
LIMIT = 100000  # @param {type:"integer"}

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=DAYS)

print(f"📅 Fetching transactions from {start_date.date()} to {end_date.date()}")
print(f"⚙️ Filters: min value={MIN_VALUE_ETH} ETH, sample rate={SAMPLE_RATE*100}%")

✅ Fetcher initialized


In [ ]:
# Build the query
query = f"""
SELECT
    `hash` as tx_hash,
    block_number,
    block_timestamp,
    from_address,
    to_address,
    CAST(value AS FLOAT64) / 1e18 as value_eth,
    CAST(gas_price AS FLOAT64) / 1e9 as gas_price_gwei,
    receipt_gas_used as gas_used,
    gas as gas_limit
FROM `bigquery-public-data.crypto_ethereum.transactions`
WHERE 
    block_timestamp >= TIMESTAMP('{start_date.strftime('%Y-%m-%d')}')
    AND block_timestamp < TIMESTAMP('{end_date.strftime('%Y-%m-%d')}')
    AND to_address IS NOT NULL
    AND CAST(value AS FLOAT64) / 1e18 >= {MIN_VALUE_ETH}
    AND RAND() < {SAMPLE_RATE}
ORDER BY block_timestamp DESC
LIMIT {LIMIT}
"""

# Dry run to estimate cost
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run = client.query(query, job_config=job_config)

gb_processed = dry_run.total_bytes_processed / (1024 ** 3)
cost_estimate = gb_processed * 5 / 1000  # $5 per TB

print(f"📊 Query will process: {gb_processed:.2f} GB")
print(f"💰 Estimated cost: ${cost_estimate:.4f} (first 1TB/month is FREE)")
print(f"\n✅ Ready to run. Execute next cell to fetch data.")

⏳ Fetching ~5,000 transactions...
   Rate limit: 5.0s between requests

  [1/10] Fetching from 0x28c6c062...
      Got 0 transactions
  [2/10] Fetching from 0xdfd5293d...
      Got 0 transactions
  [2/10] Fetching from 0xdfd5293d...
      Got 0 transactions
  [3/10] Fetching from 0x21a31ee1...
      Got 0 transactions
  [3/10] Fetching from 0x21a31ee1...
      Got 0 transactions
  [4/10] Fetching from 0x56eddb7a...
      Got 0 transactions
  [4/10] Fetching from 0x56eddb7a...
      Got 0 transactions
  [5/10] Fetching from 0xa9d1e08c...
      Got 0 transactions
  [5/10] Fetching from 0xa9d1e08c...
      Got 0 transactions
  [6/10] Fetching from 0x7a250d56...
      Got 0 transactions
  [6/10] Fetching from 0x7a250d56...
      Got 0 transactions
  [7/10] Fetching from 0x3fc91a3a...
      Got 0 transactions
  [7/10] Fetching from 0x3fc91a3a...
      Got 0 transactions
  [8/10] Fetching from 0xdef1c0de...
      Got 0 transactions
  [8/10] Fetching from 0xdef1c0de...
      Got 0 transaction

In [ ]:
# Run the query and fetch data
import time

print("⏳ Fetching transactions from BigQuery...")
start_time = time.time()

df = client.query(query).to_dataframe()

elapsed = time.time() - start_time
print(f"✅ Fetched {len(df):,} transactions in {elapsed:.1f}s")
df.head()

In [ ]:
# Data overview
print(f"\n📈 Transaction Statistics:")
print(f"  Total transactions: {len(df):,}")
print(f"  Unique senders: {df['from_address'].nunique():,}")
print(f"  Unique receivers: {df['to_address'].nunique():,}")
print(f"  Total ETH transferred: {df['value_eth'].sum():,.2f}")
print(f"  Avg transaction: {df['value_eth'].mean():.4f} ETH")
print(f"  Max transaction: {df['value_eth'].max():,.2f} ETH")
print(f"  Date range: {df['block_timestamp'].min()} to {df['block_timestamp'].max()}")

## 3️⃣a FULL Data Export (No Sampling)

🚀 **Download the complete Ethereum dataset** - no LIMIT, no sampling!

Uses day-by-day processing to handle millions of rows efficiently:
- Processes one day at a time to manage memory
- Saves each day as a separate Parquet file
- Supports resume (skips already downloaded days)
- Progress tracking with estimates

In [ ]:
# ============================================================================
# FULL DATA EXPORT CONFIGURATION
# ============================================================================

# Date range for full export
FULL_EXPORT_DAYS = 7  # @param {type:"integer"} Number of days to download
FULL_EXPORT_START = ""  # @param {type:"string"} Optional: Start date (YYYY-MM-DD)
FULL_EXPORT_END = ""    # @param {type:"string"} Optional: End date (YYYY-MM-DD)

# Filtering options
MIN_VALUE_ETH_FULL = 0.0  # @param {type:"number"} Set > 0 to filter dust transactions
INCLUDE_CONTRACT_CALLS = True  # @param {type:"boolean"} Include transactions to contracts

# Output settings
FULL_EXPORT_DIR = os.path.join(DRIVE_PROJECT_DIR, "full_export")

# Calculate date range
from datetime import datetime, timedelta

if FULL_EXPORT_START and FULL_EXPORT_END:
    full_start_date = datetime.strptime(FULL_EXPORT_START, '%Y-%m-%d')
    full_end_date = datetime.strptime(FULL_EXPORT_END, '%Y-%m-%d')
else:
    full_end_date = datetime.utcnow()
    full_start_date = full_end_date - timedelta(days=FULL_EXPORT_DAYS)

print(f"📅 Full Export Date Range: {full_start_date.date()} to {full_end_date.date()}")
print(f"⚙️ Min ETH filter: {MIN_VALUE_ETH_FULL}")
print(f"📂 Output directory: {FULL_EXPORT_DIR}")
print(f"\n✅ Configuration ready. Run next cell to estimate data size.")

In [ ]:
# ============================================================================
# ESTIMATE DATA SIZE (run this first!)
# ============================================================================

def estimate_full_export(client, start_date, end_date, min_value_eth=0.0):
    """Estimate the size and cost of full data export."""
    
    min_value_clause = f"AND CAST(value AS FLOAT64) / 1e18 >= {min_value_eth}" if min_value_eth > 0 else ""
    
    query = f"""
    SELECT 
        COUNT(*) as total_transactions,
        COALESCE(SUM(CAST(value AS FLOAT64)) / 1e18, 0) as total_eth,
        COUNT(DISTINCT from_address) as unique_senders,
        COUNT(DISTINCT to_address) as unique_receivers
    FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE 
        block_timestamp >= TIMESTAMP('{start_date.strftime('%Y-%m-%d')}')
        AND block_timestamp < TIMESTAMP('{end_date.strftime('%Y-%m-%d')}')
        AND to_address IS NOT NULL
        {min_value_clause}
    """
    
    # Dry run for cost estimate
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    dry_run = client.query(query, job_config=job_config)
    gb_scanned = dry_run.total_bytes_processed / (1024**3)
    
    # Run actual count query
    print("📊 Estimating data size...")
    result = client.query(query).result()
    row = list(result)[0]
    
    estimate = {
        "total_transactions": row.total_transactions,
        "total_eth": round(row.total_eth or 0, 2),
        "unique_senders": row.unique_senders,
        "unique_receivers": row.unique_receivers,
        "gb_to_scan": round(gb_scanned, 2),
        "cost_usd": round(gb_scanned * 0.005, 4),  # $5/TB
        "file_size_gb": round(row.total_transactions * 500 / (1024**3), 2),  # ~500 bytes/row
    }
    
    return estimate

# Run estimation
estimate = estimate_full_export(client, full_start_date, full_end_date, MIN_VALUE_ETH_FULL)

print("\n" + "=" * 60)
print("📦 FULL EXPORT ESTIMATE")
print("=" * 60)
print(f"📈 Total transactions:  {estimate['total_transactions']:,}")
print(f"💰 Total ETH moved:     {estimate['total_eth']:,.2f}")
print(f"👥 Unique senders:      {estimate['unique_senders']:,}")
print(f"👤 Unique receivers:    {estimate['unique_receivers']:,}")
print("-" * 60)
print(f"📊 BigQuery scan size:  {estimate['gb_to_scan']:.2f} GB")
print(f"💵 Estimated cost:      ${estimate['cost_usd']:.4f} (first 1TB/month FREE)")
print(f"💾 Output file size:    ~{estimate['file_size_gb']:.2f} GB")
print("=" * 60)
print("\n✅ Review the estimate above, then run next cell to start download.")

In [ ]:
# ============================================================================
# FULL EXPORT - DAY BY DAY PROCESSING
# ============================================================================
import os
import json
import time
from pathlib import Path

class BigQueryFullExporter:
    """
    Export full Ethereum transaction data from BigQuery.
    
    Features:
    - Day-by-day processing to manage memory
    - Parquet output with Snappy compression
    - Resume capability (skips already downloaded days)
    - Progress tracking
    """
    
    def __init__(self, client, output_dir):
        self.client = client
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def export_day(self, date, min_value_eth=0.0):
        """Export all transactions for a single day."""
        date_str = date.strftime('%Y-%m-%d')
        next_date = date + timedelta(days=1)
        next_date_str = next_date.strftime('%Y-%m-%d')
        
        output_file = self.output_dir / f"eth_transactions_{date_str}.parquet"
        
        # Skip if already exists (resume support)
        if output_file.exists():
            existing_df = pd.read_parquet(output_file)
            print(f"  ⏭️ {date_str}: Already exists ({len(existing_df):,} rows) - skipping")
            return len(existing_df), True  # rows, was_skipped
        
        min_value_clause = f"AND CAST(value AS FLOAT64) / 1e18 >= {min_value_eth}" if min_value_eth > 0 else ""
        
        query = f"""
        SELECT
            `hash` as tx_hash,
            block_number,
            block_timestamp,
            from_address,
            to_address,
            CAST(value AS FLOAT64) / 1e18 as value_eth,
            CAST(gas_price AS FLOAT64) / 1e9 as gas_price_gwei,
            receipt_gas_used as gas_used,
            gas as gas_limit,
            COALESCE(transaction_type, 0) as transaction_type,
            nonce,
            transaction_index
        FROM `bigquery-public-data.crypto_ethereum.transactions`
        WHERE 
            block_timestamp >= TIMESTAMP('{date_str}')
            AND block_timestamp < TIMESTAMP('{next_date_str}')
            AND to_address IS NOT NULL
            {min_value_clause}
        ORDER BY block_number, transaction_index
        """
        
        print(f"  ⏳ {date_str}: Fetching...")
        start_time = time.time()
        
        df = self.client.query(query).to_dataframe()
        
        if len(df) == 0:
            print(f"  ⚠️ {date_str}: No data found")
            return 0, False
        
        # Save to Parquet with compression
        df.to_parquet(output_file, compression='snappy', index=False)
        
        elapsed = time.time() - start_time
        file_size_mb = output_file.stat().st_size / (1024**2)
        print(f"  ✅ {date_str}: Saved {len(df):,} transactions ({file_size_mb:.1f} MB) in {elapsed:.1f}s")
        
        return len(df), False
    
    def export_range(self, start_date, end_date, min_value_eth=0.0):
        """Export all transactions in date range, one day at a time."""
        print("\n" + "=" * 60)
        print("🚀 STARTING FULL EXPORT")
        print("=" * 60)
        print(f"📅 Date range: {start_date.date()} to {end_date.date()}")
        print(f"📂 Output: {self.output_dir}")
        print()
        
        current_date = start_date
        total_rows = 0
        days_processed = 0
        days_skipped = 0
        overall_start = time.time()
        
        total_days = (end_date - start_date).days
        
        while current_date < end_date:
            rows, was_skipped = self.export_day(current_date, min_value_eth)
            total_rows += rows
            days_processed += 1
            if was_skipped:
                days_skipped += 1
            current_date += timedelta(days=1)
            
            # Progress update
            progress = days_processed / total_days * 100
            print(f"  📊 Progress: {progress:.0f}% ({days_processed}/{total_days} days)")
        
        elapsed = time.time() - overall_start
        
        # Create manifest file
        manifest = {
            "export_date": datetime.now().isoformat(),
            "start_date": start_date.strftime('%Y-%m-%d'),
            "end_date": end_date.strftime('%Y-%m-%d'),
            "days_processed": days_processed,
            "days_skipped": days_skipped,
            "total_rows": total_rows,
            "min_value_eth": min_value_eth,
            "elapsed_seconds": round(elapsed, 1),
            "files": sorted([f.name for f in self.output_dir.glob("eth_transactions_*.parquet")])
        }
        
        with open(self.output_dir / "manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        
        print("\n" + "=" * 60)
        print("✅ EXPORT COMPLETE")
        print("=" * 60)
        print(f"📈 Total rows:     {total_rows:,}")
        print(f"📅 Days processed: {days_processed} ({days_skipped} skipped/resumed)")
        print(f"⏱️ Total time:     {elapsed/60:.1f} minutes")
        print(f"📂 Output:         {self.output_dir}")
        print(f"📋 Manifest:       {self.output_dir / 'manifest.json'}")
        
        return manifest

# Initialize exporter
exporter = BigQueryFullExporter(client, FULL_EXPORT_DIR)

# Start the full export!
manifest = exporter.export_range(full_start_date, full_end_date, MIN_VALUE_ETH_FULL)

In [ ]:
# ============================================================================
# LOAD ALL PARQUET FILES INTO SINGLE DATAFRAME
# ============================================================================

def load_full_export(export_dir):
    """Load all exported Parquet files into a single DataFrame."""
    export_path = Path(export_dir)
    parquet_files = sorted(export_path.glob("eth_transactions_*.parquet"))
    
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {export_dir}")
    
    print(f"📂 Loading {len(parquet_files)} Parquet files...")
    
    dfs = []
    total_size = 0
    
    for pf in parquet_files:
        df_chunk = pd.read_parquet(pf)
        dfs.append(df_chunk)
        total_size += pf.stat().st_size
        print(f"  ✅ {pf.name}: {len(df_chunk):,} rows")
    
    df_full = pd.concat(dfs, ignore_index=True)
    
    print("\n" + "=" * 60)
    print("📊 FULL DATASET LOADED")
    print("=" * 60)
    print(f"📈 Total rows:       {len(df_full):,}")
    print(f"💾 Total file size:  {total_size / (1024**3):.2f} GB")
    print(f"📅 Date range:       {df_full['block_timestamp'].min()} to {df_full['block_timestamp'].max()}")
    print(f"👥 Unique senders:   {df_full['from_address'].nunique():,}")
    print(f"👤 Unique receivers: {df_full['to_address'].nunique():,}")
    print(f"💰 Total ETH:        {df_full['value_eth'].sum():,.2f}")
    
    return df_full

# Load the full export into df_full
df_full = load_full_export(FULL_EXPORT_DIR)

# Optionally replace df with df_full for downstream cells
# df = df_full

In [ ]:
# ============================================================================
# DOWNLOAD FULL EXPORT TO LOCAL MACHINE
# ============================================================================
import shutil

def download_full_export(export_dir, create_zip=True):
    """Create a zip of all export files and provide download link."""
    export_path = Path(export_dir)
    
    if create_zip:
        zip_name = f"eth_full_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        zip_path = export_path.parent / zip_name
        
        print(f"📦 Creating zip archive...")
        shutil.make_archive(str(zip_path), 'zip', export_path)
        
        zip_file = f"{zip_path}.zip"
        zip_size = os.path.getsize(zip_file) / (1024**3)
        print(f"✅ Created: {zip_file} ({zip_size:.2f} GB)")
        
        # Try Colab download
        try:
            from google.colab import files
            print("📥 Starting download...")
            files.download(zip_file)
        except:
            # VS Code / local environment
            print(f"\n📂 Download the zip file from:")
            print(f"   {os.path.abspath(zip_file)}")
            from IPython.display import FileLink, display
            display(FileLink(zip_file, result_html_prefix="📥 Download: "))
    else:
        # List individual files for manual download
        parquet_files = sorted(export_path.glob("*.parquet"))
        print(f"📂 Export files in {export_path}:")
        for pf in parquet_files:
            size_mb = pf.stat().st_size / (1024**2)
            print(f"   - {pf.name} ({size_mb:.1f} MB)")

# Download the full export
download_full_export(FULL_EXPORT_DIR, create_zip=True)

### 💡 Tips for Large Exports

**Memory Management:**
- Day-by-day processing keeps memory under control
- Each day's data is saved separately, then combined when loading

**Resume Support:**
- If the export is interrupted, just re-run - it will skip already downloaded days
- Check `manifest.json` for export status

**For Very Large Exports (months of data):**
- Consider running overnight
- Use `MIN_VALUE_ETH_FULL > 0` to filter dust transactions
- Process in weekly batches if needed

**Cost Control:**
- First 1 TB/month of BigQuery scans is FREE
- Ethereum transactions table: ~5-10 GB per day
- 7 days ≈ 35-70 GB ≈ $0.17-0.35 if over free tier

## 3️⃣b Save & Reload as Parquet
- Saves the 7-day slice locally in Parquet (compressed)
- Lets you re-load quickly or download to your machine

In [ ]:
# Save to Parquet - works in both Colab Web and VS Code
import os
from IPython.display import display, HTML, FileLink

PARQUET_FILE = "eth_last_7_days.parquet"
COMPRESSION = "snappy"

def save_and_download(df, filename, output_dir):
    """Save file and provide download - works in Colab Web and VS Code."""
    # Save to output directory
    filepath = os.path.join(output_dir, filename)
    
    if filename.endswith('.parquet'):
        df.to_parquet(filepath, index=False, compression=COMPRESSION)
    else:
        df.to_csv(filepath, index=False)
    
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"✅ Saved {len(df):,} rows to {filepath} ({size_mb:.2f} MB)")
    
    # Try Colab download first
    try:
        from google.colab import files
        files.download(filepath)
        print("📥 Download started (Colab)")
    except:
        # VS Code: Show clickable link
        print(f"📂 File saved to: {os.path.abspath(filepath)}")
        display(FileLink(filepath, result_html_prefix="📥 Download: "))

if 'df' in globals() and len(df) > 0:
    save_and_download(df, PARQUET_FILE, DRIVE_PROJECT_DIR)
else:
    raise RuntimeError("DataFrame df not found. Run the fetch step first.")

## 3️⃣c Load from Local Parquet (Alternative)
If you already have a Parquet file or want to skip BigQuery, load data directly:

In [ ]:
# Load from local downloaded files (skip BigQuery)
import pandas as pd
import os

# Your downloaded files
PARQUET_FILE = r"C:\Users\Administrator\Downloads\eth_last_7_days.parquet"
CSV_FILE = r"C:\Users\Administrator\Downloads\InteractiveSheet_2025-12-21_18_40_55 - Sheet1.csv"

# Choose which file to load
USE_PARQUET = True  # @param {type:"boolean"} Set False to use CSV instead

if USE_PARQUET and os.path.exists(PARQUET_FILE):
    df = pd.read_parquet(PARQUET_FILE)
    print(f"✅ Loaded {len(df):,} rows from Parquet")
    print(f"📂 Source: {PARQUET_FILE}")
elif os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    print(f"✅ Loaded {len(df):,} rows from CSV")
    print(f"📂 Source: {CSV_FILE}")
else:
    raise FileNotFoundError(f"Neither file found:\n  - {PARQUET_FILE}\n  - {CSV_FILE}")

print(f"\n📊 Columns: {list(df.columns)}")
print(f"📈 Date range: {df['block_timestamp'].min()} to {df['block_timestamp'].max()}")
df.head()

## 4️⃣ Connect to Memgraph

Options:
- **Local**: Use ngrok to expose your local Memgraph
- **Cloud**: Use Memgraph Cloud (free tier available)
- **Docker in Colab**: Run Memgraph directly in Colab (limited)

In [ ]:
# Option A: Connect to remote Memgraph (local via ngrok or cloud)
MEMGRAPH_HOST = "localhost"  # @param {type:"string"}
MEMGRAPH_PORT = 7687  # @param {type:"integer"}

# For ngrok tunnel from your local machine:
# 1. Install ngrok: https://ngrok.com/download
# 2. Run: ngrok tcp 7687
# 3. Copy the forwarding address (e.g., 0.tcp.ngrok.io:12345)
USE_NGROK = False  # @param {type:"boolean"}
NGROK_URL = "0.tcp.ngrok.io"  # @param {type:"string"}
NGROK_PORT = 12345  # @param {type:"integer"}

if USE_NGROK:
    MEMGRAPH_HOST = NGROK_URL
    MEMGRAPH_PORT = NGROK_PORT

In [ ]:
# Try connecting with native pymgclient (preferred for Memgraph)
import mgclient

def test_memgraph_connection(host, port):
    """Test connection using native pymgclient driver."""
    try:
        conn = mgclient.connect(host=host, port=port)
        cursor = conn.cursor()
        cursor.execute("RETURN 1 as test")
        cursor.fetchall()
        conn.close()
        return True, "Connected via pymgclient"
    except Exception as e:
        return False, str(e)

connected, msg = test_memgraph_connection(MEMGRAPH_HOST, MEMGRAPH_PORT)
if connected:
    print(f"✅ Connected to Memgraph at {MEMGRAPH_HOST}:{MEMGRAPH_PORT} (native driver)")
else:
    print(f"❌ Connection failed: {msg}")
    print("\n💡 Options:")
    print("  1. Run ngrok on your local machine: ngrok tcp 7687")
    print("  2. Use Memgraph Cloud: https://cloud.memgraph.com")
    print("  3. Run Memgraph in Colab (see next cell)")

In [ ]:
# Option B: Run Memgraph directly in Colab (experimental)
# Set to False - we're using local Memgraph on your laptop instead
RUN_MEMGRAPH_IN_COLAB = False  # @param {type:"boolean"}

if RUN_MEMGRAPH_IN_COLAB:
    print("⚠️ Skipped - Using local Memgraph instead")
    print("   Set RUN_MEMGRAPH_IN_COLAB = True if you want to run Memgraph in Colab")
else:
    print("✅ Using local Memgraph on your laptop")
    print("   Make sure Docker containers are running:")
    print("   - memgraph/memgraph-mage on port 7687")
    print("   - memgraph/lab on port 3000")
    print()
    print("   To start them:")
    print("   docker run -d -p 7687:7687 -p 7444:7444 --name memgraph memgraph/memgraph-mage")
    print("   docker run -d -p 3000:3000 -e QUICK_CONNECT_MG_HOST=host.docker.internal --name lab memgraph/lab")

## 5️⃣ Ingest Data into Memgraph

In [ ]:
import mgclient
import requests
import pandas as pd

class MemgraphIngester:
    """Ingest transactions into Memgraph using native pymgclient driver."""
    
    def __init__(self, host, port):
        self.host = host
        self.port = port
        self.conn = mgclient.connect(host=host, port=port)
        self.known_addresses = {}
    
    def fetch_sanctioned_addresses(self):
        """Fetch real OFAC sanctioned Ethereum addresses."""
        print("📥 Fetching OFAC sanctioned addresses...")
        
        # OFAC SDN List - Ethereum addresses (from Chainalysis public data)
        ofac_addresses = [
            # Tornado Cash (sanctioned Aug 2022)
            "0x8589427373d6d84e98730d7795d8f6f8731fda16",  # Tornado Cash: Router
            "0x722122df12d4e14e13ac3b6895a86e84145b6967",  # Tornado Cash: Proxy
            "0xdd4c48c0b24039969fc16d1cdf626eab821d3384",  # Tornado Cash: 0.1 ETH
            "0xd90e2f925da726b50c4ed8d0fb90ad053324f31b",  # Tornado Cash: 1 ETH
            "0x4736dcf1b7a3d580672cce6e7c65cd5cc9cfba9d",  # Tornado Cash: 10 ETH
            "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf",  # Tornado Cash: 100 ETH
            "0xa160cdab225685da1d56aa342ad8841c3b53f291",  # Tornado Cash: Governance
            "0xd882cfc20f52f2599d84b8e8d58c7fb62cfe344b",  # Tornado Cash: Voucher
            "0x7f367cc41522ce07553e823bf3be79a889debe1b",  # Tornado Cash: Mining
            "0x94a1b5cdb22c43faab4abeb5c74999895464ddba",  # Tornado Cash: 100 DAI
            "0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc",  # Tornado Cash: 1000 DAI
            "0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936",  # Tornado Cash: 10000 DAI
            "0x23773e65ed146a459791799d01336db287f25334",  # Tornado Cash: 100000 DAI
            "0xd691f27f38b395864ea86cfc7253969b409c362d",  # Tornado Cash: Relayer
            "0x22aaa7720ddd5388a3c0a3333430953c68f1849b",  # Tornado Cash: Relayer 2
            "0x03893a7c7463ae47d46bc7f091665f1893656003",  # Tornado Cash: Gitcoin
            "0x179f48c78f57a3a78f0608cc9197b8972921d1d2",  # Tornado Cash: Deposit
            "0xb1c8094b234dce6e03f10a5b673c1d8c69739a00",  # Tornado Cash
            "0x527653ea119f3e6a1f5bd18fbf4714081d7b31ce",  # Tornado Cash
            "0x58e8dcc13be9780fc42e8723d8ead4cf46943df2",  # Tornado Cash: Router 2
            "0xd96f2b1c14db8458374d9aca76e26c3d18364307",  # Tornado Cash: 0.1 WBTC
            "0x178169b423a011fff22b9e3f3abea13414ddd0f1",  # Tornado Cash: Nova
            "0x610b717796ad172b316836ac95a2ffad065ceab4",  # Tornado Cash: Nova Proxy
            "0xdf231d99ff8b6c6cbf4e9b9a945cbacef9339178",  # Tornado Cash
            "0xba214c1c1928a32bffe790263e38b4af9bfcd659",  # Tornado Cash
            "0xb6f5ec1a0a9cd1526536d3f0426c429529471f40",  # Tornado Cash
            "0x8576acc5c05d6ce88f4e49bf65bdf0c62f91353c",  # Tornado Cash
            "0xaeaac358560e11f52454d997aaff2c5731b6f8a6",  # Tornado Cash
            "0x1356c899d8c9467c7f71c195612f8a395abf2f0a",  # Tornado Cash
            "0xa60c772958a3ed56c1f15dd055ba37ac8e523a0d",  # Tornado Cash
            "0x169ad27a470d064dede56a2d3ff727986b15d52b",  # Tornado Cash
            "0x0836222f2b2b24a3f36f98668ed8f0b38d1a872f",  # Tornado Cash
            "0xf67721a2d8f736e75a49fdd7fad2e31d8676542a",  # Tornado Cash
            "0x9ad122c22b14202b4490edaf288fdb3c7cb3ff5e",  # Tornado Cash
            "0x905b63fff465b9ffbf41dea908ceb12478ec7601",  # Tornado Cash
            "0x07687e702b410fa43f4cb4af7fa097918ffd2730",  # Tornado Cash
            "0x94c92f096437ab9958fc0a37f09348f30389ae79",  # Tornado Cash
            "0x3aac1cc67c2ec5db4ea850957b967ba153ad6279",  # Tornado Cash
            "0x723b78e67497e85279cb204544566f4dc5d2aca0",  # Tornado Cash
            "0xcc84179ffd19a1627e79f8648d09e095252bc418",  # Tornado Cash
            # Blender.io (sanctioned May 2022)
            "0x8589427373d6d84e98730d7795d8f6f8731fda16",  # Blender
            # Lazarus Group / North Korea
            "0x098b716b8aaf21512996dc57eb0615e2383e2f96",  # Lazarus Group
            "0xa0e1c89ef1a489c9c7de96311ed5ce5d32c20e4b",  # Lazarus Group
            "0x3cffd56b47b7b41c56258d9c7731abadc360e073",  # Lazarus Group
            "0x53b6936513e738f44fb50d2b9476730c0ab3bfc1",  # Lazarus Group
            # Garantex (Russian exchange)
            "0x6f1ca141a28907f78ebaa64fb83a9088b02a8352",  # Garantex
        ]
        
        for addr in ofac_addresses:
            self.known_addresses[addr.lower()] = ("Sanctioned", "OFAC")
        
        print(f"  ✅ Loaded {len(ofac_addresses)} OFAC sanctioned addresses")
        return len(ofac_addresses)
    
    def fetch_mixer_addresses(self):
        """Fetch known mixer/tumbler addresses."""
        print("📥 Fetching mixer addresses...")
        
        mixer_addresses = [
            # Tornado Cash (also mixers)
            "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf",
            "0xd90e2f925da726b50c4ed8d0fb90ad053324f31b", 
            "0x4736dcf1b7a3d580672cce6e7c65cd5cc9cfba9d",
            "0xdd4c48c0b24039969fc16d1cdf626eab821d3384",
            # ChipMixer
            "0x57d90b64a1a57749b0f932f1a3395792e12e7055",
            # Wasabi Wallet (CoinJoin)
            "0x8589427373d6d84e98730d7795d8f6f8731fda16",
            # Other known mixers
            "0xd882cfc20f52f2599d84b8e8d58c7fb62cfe344b",
            "0x7f367cc41522ce07553e823bf3be79a889debe1b",
        ]
        
        for addr in mixer_addresses:
            if addr.lower() not in self.known_addresses:
                self.known_addresses[addr.lower()] = ("Mixer", "Known Mixer")
        
        print(f"  ✅ Loaded {len(mixer_addresses)} mixer addresses")
        return len(mixer_addresses)
    
    def fetch_exchange_addresses(self):
        """Fetch major exchange hot wallet addresses."""
        print("📥 Fetching exchange addresses...")
        
        exchange_addresses = {
            # Binance
            "0x28c6c06298d514db089934071355e5743bf21d60": "Binance",
            "0x21a31ee1afc51d94c2efccaa2092ad1028285549": "Binance",
            "0xdfd5293d8e347dfe59e90efd55b2956a1343963d": "Binance",
            "0x56eddb7aa87536c09ccc2793473599fd21a8b17f": "Binance",
            "0x9696f59e4d72e237be84ffd425dcad154bf96976": "Binance",
            "0xf977814e90da44bfa03b6295a0616a897441acec": "Binance",
            "0x5a52e96bacdabb82fd05763e25335261b270efcb": "Binance",
            "0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be": "Binance",
            # Coinbase
            "0x71660c4005ba85c37ccec55d0c4493e66fe775d3": "Coinbase",
            "0x503828976d22510aad0201ac7ec88293211d23da": "Coinbase",
            "0xddfabcdc4d8ffc6d5beaf154f18b778f892a0740": "Coinbase",
            "0x3cd751e6b0078be393132286c442345e5dc49699": "Coinbase",
            "0xa9d1e08c7793af67e9d92fe308d5697fb81d3e43": "Coinbase",
            # Kraken
            "0x2910543af39aba0cd09dbb2d50200b3e800a63d2": "Kraken",
            "0x0a869d79a7052c7f1b55a8ebabbea3420f0d1e13": "Kraken",
            "0xe853c56864a2ebe4576a807d26fdc4a0ada51919": "Kraken",
            # FTX (defunct but historical)
            "0x2faf487a4414fe77e2327f0bf4ae2a264a776ad2": "FTX",
            "0xc098b2a3aa256d2140208c3de6543aaef5cd3a94": "FTX",
            # Huobi
            "0xab5c66752a9e8167967685f1450532fb96d5d24f": "Huobi",
            "0x6748f50f686bfbca6fe8ad62b22228b87f31ff2b": "Huobi",
            "0xfdb16996831753d5331ff813c29a93c76834a0ad": "Huobi",
            # OKX
            "0x6cc5f688a315f3dc28a7781717a9a798a59fda7b": "OKX",
            "0x236f9f97e0e62388479bf9e5ba4889e46b0273c3": "OKX",
            # Bitfinex
            "0x876eabf441b2ee5b5b0554fd502a8e0600950cfa": "Bitfinex",
            "0x742d35cc6634c0532925a3b844bc454e4438f44e": "Bitfinex",
            # Gemini
            "0xd24400ae8bfebb18ca49be86258a3c749cf46853": "Gemini",
            "0x6fc82a5fe25a5cdb58bc74600a40a69c065263f8": "Gemini",
        }
        
        for addr, name in exchange_addresses.items():
            self.known_addresses[addr.lower()] = ("Exchange", name)
        
        print(f"  ✅ Loaded {len(exchange_addresses)} exchange addresses")
        return len(exchange_addresses)
    
    def fetch_defi_addresses(self):
        """Fetch major DeFi protocol addresses."""
        print("📥 Fetching DeFi protocol addresses...")
        
        defi_addresses = {
            # Uniswap
            "0x7a250d5630b4cf539739df2c5dacb4c659f2488d": "Uniswap V2 Router",
            "0xe592427a0aece92de3edee1f18e0157c05861564": "Uniswap V3 Router",
            "0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45": "Uniswap V3 Router 2",
            "0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad": "Uniswap Universal Router",
            # Aave
            "0x7d2768de32b0b80b7a3454c06bdac94a69ddc7a9": "Aave V2 Lending Pool",
            "0x87870bca3f3fd6335c3f4ce8392d69350b4fa4e2": "Aave V3 Pool",
            # Compound
            "0x3d9819210a31b4961b30ef54be2aed79b9c9cd3b": "Compound Comptroller",
            # Curve
            "0xbebc44782c7db0a1a60cb6fe97d0b483032ff1c7": "Curve 3pool",
            "0xd51a44d3fae010294c616388b506acda1bfaae46": "Curve Tricrypto",
            # 1inch
            "0x1111111254fb6c44bac0bed2854e76f90643097d": "1inch Router",
            "0x1111111254eeb25477b68fb85ed929f73a960582": "1inch V5 Router",
            # SushiSwap
            "0xd9e1ce17f2641f24ae83637ab66a2cca9c378b9f": "SushiSwap Router",
            # OpenSea
            "0x00000000006c3852cbef3e08e8df289169ede581": "OpenSea Seaport",
            # Lido
            "0xae7ab96520de3a18e5e111b5eaab095312d7fe84": "Lido stETH",
        }
        
        for addr, name in defi_addresses.items():
            self.known_addresses[addr.lower()] = ("DeFi", name)
        
        print(f"  ✅ Loaded {len(defi_addresses)} DeFi addresses")
        return len(defi_addresses)
    
    def load_all_known_addresses(self):
        """Load all known address categories."""
        print("\n🔍 Loading known addresses from public sources...")
        print("=" * 50)
        
        total = 0
        total += self.fetch_sanctioned_addresses()
        total += self.fetch_mixer_addresses()
        total += self.fetch_exchange_addresses()
        total += self.fetch_defi_addresses()
        
        print("=" * 50)
        print(f"📊 Total known addresses: {len(self.known_addresses)}")
        
        # Summary by category
        categories = {}
        for addr, (cat, name) in self.known_addresses.items():
            categories[cat] = categories.get(cat, 0) + 1
        for cat, count in sorted(categories.items()):
            print(f"  • {cat}: {count}")
        
        return self.known_addresses
    
    def _execute(self, query, params=None):
        """Execute a Cypher query."""
        cursor = self.conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        self.conn.commit()
        return cursor
    
    def _fetch_one(self, query):
        """Execute query and fetch one result."""
        cursor = self.conn.cursor()
        cursor.execute(query)
        result = cursor.fetchone()
        return result[0] if result else None
    
    def setup_schema(self):
        self._execute("CREATE INDEX ON :Address(id)")
        print("✅ Schema created")
    
    def add_known_addresses(self):
        """Add all known addresses to the graph with labels."""
        if not self.known_addresses:
            self.load_all_known_addresses()
        
        print(f"\n📤 Adding {len(self.known_addresses)} known addresses to graph...")
        added = 0
        for addr, (label, name) in self.known_addresses.items():
            query = f"MERGE (a:Address {{id: $addr}}) SET a:{label}, a.name = $name"
            self._execute(query, {"addr": addr, "name": name})
            added += 1
            if added % 20 == 0:
                print(f"  Progress: {added}/{len(self.known_addresses)}")
        
        print(f"✅ Added {added} known addresses with labels")
    
    def ingest_batch(self, transactions_df, batch_size=1000):
        total = len(transactions_df)
        ingested = 0
        
        for i in range(0, total, batch_size):
            batch = transactions_df.iloc[i:i+batch_size]
            
            # Insert each transaction
            for _, row in batch.iterrows():
                from_addr = row['from_address'].lower()
                to_addr = row['to_address'].lower()
                ts = int(row['block_timestamp'].timestamp()) if hasattr(row['block_timestamp'], 'timestamp') else 0
                
                query = """
                MERGE (from:Address {id: $from_addr})
                MERGE (to:Address {id: $to_addr})
                CREATE (from)-[:TRANSFER {
                    tx_hash: $tx_hash,
                    value: $value,
                    ts: $ts,
                    block: $block
                }]->(to)
                """
                self._execute(query, {
                    "from_addr": from_addr,
                    "to_addr": to_addr,
                    "tx_hash": row['tx_hash'],
                    "value": float(row['value_eth']),
                    "ts": ts,
                    "block": int(row['block_number'])
                })
            
            ingested += len(batch)
            if (i // batch_size) % 10 == 0:
                print(f"  Progress: {ingested:,}/{total:,} ({ingested/total*100:.1f}%)")
        
        print(f"✅ Ingested {ingested:,} transactions")
        return ingested
    
    def get_stats(self):
        nodes = self._fetch_one("MATCH (n) RETURN count(n) as cnt")
        edges = self._fetch_one("MATCH ()-[r]->() RETURN count(r) as cnt")
        sanctioned = self._fetch_one("MATCH (n:Sanctioned) RETURN count(n) as cnt")
        mixers = self._fetch_one("MATCH (n:Mixer) RETURN count(n) as cnt")
        exchanges = self._fetch_one("MATCH (n:Exchange) RETURN count(n) as cnt")
        defi = self._fetch_one("MATCH (n:DeFi) RETURN count(n) as cnt")
        return {
            'nodes': nodes, 
            'edges': edges,
            'sanctioned': sanctioned,
            'mixers': mixers,
            'exchanges': exchanges,
            'defi': defi
        }
    
    def close(self):
        self.conn.close()

In [ ]:
# Ingest the data with real sanctioned/mixer addresses
if connected:
    ingester = MemgraphIngester(MEMGRAPH_HOST, MEMGRAPH_PORT)
    
    print("Setting up schema...")
    ingester.setup_schema()
    
    # Load and add real known addresses (OFAC, mixers, exchanges, DeFi)
    ingester.load_all_known_addresses()
    ingester.add_known_addresses()
    
    print(f"\n📤 Ingesting {len(df):,} transactions...")
    ingester.ingest_batch(df)
    
    stats = ingester.get_stats()
    print(f"\n" + "=" * 50)
    print(f"📊 Graph Statistics")
    print(f"=" * 50)
    print(f"  Total nodes:      {stats['nodes']:,}")
    print(f"  Total edges:      {stats['edges']:,}")
    print(f"  Sanctioned:       {stats['sanctioned']:,}")
    print(f"  Mixers:           {stats['mixers']:,}")
    print(f"  Exchanges:        {stats['exchanges']:,}")
    print(f"  DeFi protocols:   {stats['defi']:,}")
else:
    print("⚠️ Skipping ingestion - Memgraph not connected")

## 6️⃣ GPU-Accelerated Fraud Scoring

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

class GPUFraudScorer:
    """GPU-accelerated fraud scoring."""
    
    def __init__(self, use_gpu=True):
        self.use_gpu = use_gpu and torch.cuda.is_available()
        self.device = 'cuda' if self.use_gpu else 'cpu'
        self.scaler = StandardScaler()
        
        # Create XGBoost with GPU
        if self.use_gpu:
            self.model = xgb.XGBClassifier(
                tree_method='hist',
                device='cuda',
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
            )
        else:
            self.model = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=6,
            )
        
        print(f"Scorer initialized (device: {self.device})")
    
    def extract_features(self, df):
        """Extract features from transaction DataFrame."""
        features = pd.DataFrame()
        
        # Transaction features
        features['value_eth'] = df['value_eth']
        features['gas_price_gwei'] = df['gas_price_gwei']
        features['gas_used'] = df['gas_used']
        features['gas_limit'] = df['gas_limit']
        features['gas_efficiency'] = df['gas_used'] / df['gas_limit'].replace(0, 1)
        
        # Value-based features
        features['log_value'] = np.log1p(df['value_eth'])
        features['is_round_value'] = (df['value_eth'] % 1 == 0).astype(int)
        
        # Sender aggregations
        sender_stats = df.groupby('from_address').agg({
            'value_eth': ['count', 'sum', 'mean', 'std'],
            'to_address': 'nunique'
        }).reset_index()
        sender_stats.columns = ['from_address', 'sender_tx_count', 'sender_total_sent', 
                                'sender_avg_value', 'sender_std_value', 'sender_unique_receivers']
        
        # Receiver aggregations
        receiver_stats = df.groupby('to_address').agg({
            'value_eth': ['count', 'sum', 'mean'],
            'from_address': 'nunique'
        }).reset_index()
        receiver_stats.columns = ['to_address', 'receiver_tx_count', 'receiver_total_received',
                                  'receiver_avg_value', 'receiver_unique_senders']
        
        # Merge back
        df_merged = df.merge(sender_stats, on='from_address', how='left')
        df_merged = df_merged.merge(receiver_stats, on='to_address', how='left')
        
        features['sender_tx_count'] = df_merged['sender_tx_count']
        features['sender_total_sent'] = df_merged['sender_total_sent']
        features['sender_avg_value'] = df_merged['sender_avg_value']
        features['sender_unique_receivers'] = df_merged['sender_unique_receivers']
        features['receiver_tx_count'] = df_merged['receiver_tx_count']
        features['receiver_total_received'] = df_merged['receiver_total_received']
        features['receiver_unique_senders'] = df_merged['receiver_unique_senders']
        
        # Fill NaN
        features = features.fillna(0)
        
        return features
    
    def score_batch(self, df):
        """Score a batch of transactions."""
        features = self.extract_features(df)
        
        # For demo, create synthetic labels based on heuristics
        # In production, you'd load a trained model
        risk_scores = np.zeros(len(features))
        
        # Heuristic scoring (replace with real model in production)
        # High value + few transactions = potential scam
        risk_scores += (features['value_eth'] > 10) * 0.2
        risk_scores += (features['sender_tx_count'] < 5) * 0.15
        risk_scores += (features['is_round_value'] == 1) * 0.1
        risk_scores += (features['sender_unique_receivers'] == 1) * 0.15
        risk_scores += (features['receiver_unique_senders'] < 3) * 0.1
        
        # Normalize
        risk_scores = np.clip(risk_scores, 0, 1)
        
        return risk_scores
    
    def get_action(self, score):
        """Determine action based on risk score."""
        if score >= 0.9:
            return 'BLOCK'
        elif score >= 0.75:
            return 'ESCROW'
        elif score >= 0.5:
            return 'REVIEW'
        elif score >= 0.25:
            return 'MONITOR'
        return 'APPROVE'

In [ ]:
# Score all transactions
import time

scorer = GPUFraudScorer(use_gpu=True)

print(f"Scoring {len(df):,} transactions...")
start = time.time()

risk_scores = scorer.score_batch(df)
df['risk_score'] = risk_scores
df['action'] = [scorer.get_action(s) for s in risk_scores]

elapsed = time.time() - start
print(f"✅ Scored in {elapsed:.2f}s ({len(df)/elapsed:.0f} tx/sec)")

In [ ]:
# Scoring results
print("\n📊 Scoring Results:")
print(df['action'].value_counts())

print(f"\n🔴 High-risk transactions (score > 0.5):")
high_risk = df[df['risk_score'] > 0.5].sort_values('risk_score', ascending=False)
print(f"  Count: {len(high_risk):,}")
print(f"  Total ETH: {high_risk['value_eth'].sum():,.2f}")

high_risk[['tx_hash', 'from_address', 'to_address', 'value_eth', 'risk_score', 'action']].head(10)

## 6️⃣b Graph-based validation with Memgraph
Cross-check the ML risk scores against graph signals (e.g., proximity to sanctioned/mixer addresses) from the ingested Memgraph graph.

In [ ]:
import pandas as pd
import mgclient

if connected:
    conn = mgclient.connect(host=MEMGRAPH_HOST, port=MEMGRAPH_PORT)
    
    def fetch_flags(cypher):
        cursor = conn.cursor()
        cursor.execute(cypher)
        rows = cursor.fetchall()
        columns = [desc.name for desc in cursor.description] if cursor.description else ["tx_hash", "reason"]
        return pd.DataFrame(rows, columns=columns) if rows else pd.DataFrame(columns=["tx_hash", "reason"])

    sanction_flags = fetch_flags(
        """
        MATCH (s:Sanctioned)-[:TRANSFER*1..2]-(:Address)-[t:TRANSFER]->(:Address)
        RETURN DISTINCT t.tx_hash AS tx_hash, 'sanction_proximity' AS reason
        LIMIT 50000
        """
    )

    mixer_flags = fetch_flags(
        """
        MATCH (m:Mixer)<-[:TRANSFER*1..2]-(:Address)-[t:TRANSFER]->(:Address)
        RETURN DISTINCT t.tx_hash AS tx_hash, 'mixer_proximity' AS reason
        LIMIT 50000
        """
    )

    graph_flags = pd.concat([sanction_flags, mixer_flags], ignore_index=True).drop_duplicates(subset=["tx_hash"])
    print(f"Graph-flagged transactions: {len(graph_flags):,}")

    df = df.merge(graph_flags, on="tx_hash", how="left")
    df['graph_flag'] = df['reason'].notna()

    overlap = df[(df['graph_flag']) & (df['risk_score'] > 0.5)]
    print(f"Overlap of graph flags with high ML risk (>0.5): {len(overlap):,}")
    print(overlap[['tx_hash', 'from_address', 'to_address', 'value_eth', 'risk_score', 'reason', 'action']].head(10))

    conn.close()
else:
    print("⚠️ Skipping graph validation - Memgraph not connected")

## 7️⃣ Expose API via ngrok (Optional)

This allows your local services to connect to Colab's scoring engine.

In [ ]:
# Set your ngrok auth token (get from https://ngrok.com)
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

if NGROK_AUTH_TOKEN:
    !ngrok config add-authtoken {NGROK_AUTH_TOKEN}
    print("✅ ngrok configured")

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Any
import uvicorn
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Create API
api = FastAPI(title="AMTTP Colab Scoring API")

class ScoreRequest(BaseModel):
    transactions: List[Dict[str, Any]]

@api.get("/health")
def health():
    return {"status": "healthy", "gpu": torch.cuda.is_available()}

@api.post("/score")
def score(request: ScoreRequest):
    df = pd.DataFrame(request.transactions)
    scores = scorer.score_batch(df)
    return {
        "scores": scores.tolist(),
        "actions": [scorer.get_action(s) for s in scores]
    }

# Start server with ngrok
if NGROK_AUTH_TOKEN:
    public_url = ngrok.connect(8000)
    print(f"\n🌐 Public API URL: {public_url}")
    print(f"  Health: {public_url}/health")
    print(f"  Score: POST {public_url}/score")
    
    # Run server (blocking)
    uvicorn.run(api, host="0.0.0.0", port=8000)

## 8️⃣ Save Results

In [ ]:
# Save scored transactions
output_file = 'scored_transactions.csv'
save_and_download(df, output_file, DRIVE_PROJECT_DIR)

In [ ]:
# Save high-risk transactions
high_risk_df = df[df['risk_score'] > 0.5]
print(f"🔴 {len(high_risk_df):,} high-risk transactions")
print(f"   Total ETH at risk: {high_risk_df['value_eth'].sum():,.2f}")

high_risk_file = 'high_risk_transactions.csv'
save_and_download(high_risk_df, high_risk_file, DRIVE_PROJECT_DIR)

---

## 🔗 Connect to Local Memgraph Lab

To visualize in Memgraph Lab on your local machine:

1. **Start Memgraph Lab locally**:
   ```bash
   docker run -p 3000:3000 -e QUICK_CONNECT_MG_HOST=localhost memgraph/lab
   ```

2. **Expose your Memgraph via ngrok**:
   ```bash
   ngrok tcp 7687
   ```

3. **Use the ngrok URL in this notebook** to ingest data

4. **Open Memgraph Lab** at http://localhost:3000

5. **Visualize fraud patterns** with queries like:
   ```cypher
   MATCH (s:Sanctioned)-[:TRANSFER*1..2]-(a:Address)
   RETURN s, a
   LIMIT 100
   ```